In [1]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
import timm
from PIL import Image
import re

# ==========================================
# 1. CONFIGURAZIONE AMBIENTE E PATH (Drive)
# ==========================================
# In Google Colab, assicurati di eseguire prima:
# from google.colab import drive
# drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Progetto_EVWSD_ML'
CACHE_DIR = os.path.join(PROJECT_DIR, 'data/hf_cache')
EMBEDDINGS_DIR = os.path.join(PROJECT_DIR, 'embeddings')

# Creiamo le cartelle di lavoro se non esistono
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)

# Impostiamo il dispositivo (GPU se disponibile, altrimenti CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo di calcolo attivo: {device}")

# ==========================================
# 2. DEFINIZIONE DEL DATASET CUSTOM (PyTorch)
# ==========================================
class EVWSDImageDataset(Dataset):
    """
    Dataset personalizzato per gestire le immagini del task EVWSD-ITA.
    Rileva automaticamente se le immagini candidate sono in una lista o divise in colonne.
    """
    def __init__(self, hf_dataset, split="train"):
        # Selezioniamo lo split corretto (train, validation o test)
        self.data = hf_dataset[split]
        self._printed_keys = False  # Per stampare le chiavi solo una volta come debug

        # Pipeline di trasformazione (estratta dal Lab 04)
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Stampa di debug per capire la struttura del dataset al primo avvio
        if not self._printed_keys:
            print("\n--- DEBUG STRUTTURA DATASET ---")
            print(f"Chiavi disponibili nel dataset: {list(item.keys())}")
            self._printed_keys = True

        candidate_images = []

        # CASO 1: Le immagini candidate sono in una colonna che contiene già una lista (es. 'images')
        if 'images' in item and isinstance(item['images'], list):
            for img in item['images']:
                if img is not None:
                    # Assicuriamoci che sia un oggetto PIL.Image
                    if isinstance(img, Image.Image):
                        img = img.convert("RGB")
                        candidate_images.append(self.transform(img))

        # CASO 2: Le immagini sono divise in colonne separate (es. 'image_0', 'image_1' o 'image_1', 'image_2'...)
        else:
            # Troviamo tutte le chiavi nel dizionario che contengono la parola 'image' o 'img'
            image_keys = [k for k in item.keys() if 'image' in k.lower() or 'img' in k.lower()]

            # Ordiniamo le chiavi numericamente per non scombinare l'ordine delle 10 immagini candidate
            def extract_number(key_string):
                numbers = re.findall(r'\d+', key_string)
                return int(numbers[0]) if numbers else 999

            image_keys = sorted(image_keys, key=extract_number)

            for k in image_keys:
                img = item[k]
                if img is not None:
                    if isinstance(img, Image.Image):
                        img = img.convert("RGB")
                        candidate_images.append(self.transform(img))

        # Controllo di sicurezza: se la lista è ancora vuota, solleviamo un errore descrittivo
        if len(candidate_images) == 0:
            raise RuntimeError(
                f"Errore: Nessuna immagine trovata all'indice {idx}. "
                f"Controlla le chiavi del dizionario: {list(item.keys())}"
            )

        # Impiliamo i tensori delle immagini candidate
        stacked_images = torch.stack(candidate_images)

        # Recuperiamo la query o l'ID per tenere traccia dell'associazione
        query_id = item.get('id', idx)

        return stacked_images, query_id

# ==========================================
# 3. CARICAMENTO MODELLO PRE-ADDESTRATO
# ==========================================
def get_feature_extractor(model_name='vit_base_patch16_224'):
    """
    Carica un modello pre-addestrato dalla libreria 'timm' (visto nel Lab 04)
    escludendo la testa di classificazione finale per usarlo come estrattore di feature.
    """
    print(f"Caricamento del modello {model_name}...")
    model = timm.create_model(model_name, pretrained=True, num_classes=0)
    model = model.to(device)
    model.eval()
    return model

# ==========================================
# 4. PIPELINE DI ESTRAZIONE E SALVATAGGIO
# ==========================================
def extract_and_save_embeddings(dataloader, model, output_filename="tensor_immagini.pt"):
    """
    Scorre l'intero dataloader, estrae le feature visive per ogni immagine candidata
    e salva il mega-tensore risultante su Google Drive.
    """
    all_embeddings = []
    all_ids = []

    with torch.no_grad():
        for batch_idx, (images_batch, ids) in enumerate(dataloader):
            # images_batch ha dimensione: [BatchSize, NumImmagini, 3, 224, 224]
            batch_size = images_batch.size(0)
            num_candidates = images_batch.size(1)

            # Appiattiamo temporaneamente per far passare tutto nel modello in un colpo solo
            flat_images = images_batch.view(-1, 3, 224, 224).to(device)

            # Estraiamo le feature
            features = model(flat_images)

            # Ripristiniamo la struttura originale dividendo nuovamente per le immagini candidate
            features_dim = features.size(-1)
            reshaped_features = features.view(batch_size, num_candidates, features_dim)

            all_embeddings.append(reshaped_features.cpu())
            all_ids.extend(ids if isinstance(ids, list) else ids.tolist())

            if (batch_idx + 1) % 10 == 0:
                print(f"Processati {batch_idx + 1}/{len(dataloader)} batch...")

    final_embeddings = torch.cat(all_embeddings, dim=0)

    save_path = os.path.join(EMBEDDINGS_DIR, output_filename)
    torch.save({
        'embeddings': final_embeddings,
        'ids': all_ids
    }, save_path)

    print(f"\nEstrazione completata con successo!")
    print(f"Tensore salvato in: {save_path}")
    print(f"Forma finale del tensore degli embedding: {final_embeddings.shape}")

# ==========================================
# 5. CODICE DI TEST PRINCIPALE
# ==========================================
if __name__ == "__main__":
    # 1. Carichiamo il dataset ufficiale EVWSD-ITA tramite Hugging Face
    print("Inizializzazione del dataset da Hugging Face...")
    hf_dataset = load_dataset("swap-uniba/EVWSD-ITA", cache_dir=CACHE_DIR)

    # 2. Creiamo l'istanza del nostro dataset PyTorch (lato Immagini)
    train_dataset = EVWSDImageDataset(hf_dataset, split="train")

    # 3. Creiamo il DataLoader di PyTorch
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=False, num_workers=2)

    # 4. Carichiamo l'estrattore di feature (Vision Transformer)
    feature_extractor = get_feature_extractor('vit_base_patch16_224')

    # 5. Avviamo l'estrazione e salviamo il file su Drive
    print("Avvio della pipeline di estrazione degli embedding...")
    extract_and_save_embeddings(train_loader, feature_extractor, output_filename="embeddings_immagini_train.pt")

Mounted at /content/drive
Dispositivo di calcolo attivo: cpu
Inizializzazione del dataset da Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/2.83k [00:00<?, ?B/s]

ds_train.json:   0%|          | 0.00/39.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Caricamento del modello vit_base_patch16_224...


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Avvio della pipeline di estrazione degli embedding...


RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_1839/2557412493.py", line 73, in __getitem__
    stacked_images = torch.stack(candidate_images)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: stack expects a non-empty TensorList
